In [ ]:
!git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/moshi-ditto-training-lip-sync-mead-1000-demo-sir-try-phase-1-oom-error.git

In [ ]:
%cd moshi-ditto-training-lip-sync-mead-1000-demo-sir-try-phase-1-oom-error

In [ ]:
!pip install huggingface_hub hf_transfer onnx rich transformers

In [ ]:
!pip uninstall -y torch torchaudio
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!bash ditto-train/scripts/setup_environment.sh

In [ ]:
# !rm -r /workspace/moshi-ditto-training-p2-lip-new-try2/ditto-train/checkpoints

In [ ]:
from huggingface_hub import snapshot_download
import os

repo_id = "Darknsu/HDTF_MEAD"
download_dir = "/workspace/"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading HDTF dataset...")

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=download_dir,
    local_dir_use_symlinks=False,
)

print("✅ File downloaded to:", download_dir)

In [ ]:
!tar -xzvf /workspace/HDTF_MEAD.tar.gz -C /workspace/

In [ ]:
from huggingface_hub import snapshot_download
import os

repo_id = "digital-avatar/ditto-talkinghead"
download_dir = "./ditto-train/checkpoints"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading full repository...")

snapshot_download(
    repo_id=repo_id,
    repo_type="model",   # important
    local_dir=download_dir,
    local_dir_use_symlinks=False,  # ensures real files, not symlinks
)

print("✅ Repo downloaded to:", download_dir)

## 📥 Download Wav2Lip SyncNet Checkpoint
Required for lip-sync loss during training. This is the pretrained "Expert Discriminator" from the Wav2Lip project (~25MB).

In [ ]:
from huggingface_hub import snapshot_download
import os

repo_id = "Nekochu/Wav2Lip"
download_dir = "./ditto-train/checkpoints"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading full repository...")

snapshot_download(
    repo_id=repo_id,
    repo_type="model",   # important
    local_dir=download_dir,
    local_dir_use_symlinks=False,  # ensures real files, not symlinks
)

print("✅ Repo downloaded to:", download_dir)

In [ ]:
!apt update -qq
!apt-get install -y ffmpeg libcudnn8 libcudnn8-dev 2>&1 | tail -5
!apt-get install -y libgles2 libgl1-mesa-glx libegl1-mesa

In [ ]:
# import os

# video_dir = "/workspace/HDTF/video"
# wav_dir = "/workspace/HDTF/wav"

# # Get filenames without extension
# video_files = {os.path.splitext(f)[0]: f for f in os.listdir(video_dir) if f.endswith(".mp4")}
# wav_files = {os.path.splitext(f)[0]: f for f in os.listdir(wav_dir) if f.endswith(".wav")}

# # Find common names
# common_names = list(set(video_files.keys()) & set(wav_files.keys()))

# # Keep only first 20 (you can shuffle if you want random)
# keep_names = set(common_names[:20])

# # Delete unwanted video files
# for name, filename in video_files.items():
#     if name not in keep_names:
#         os.remove(os.path.join(video_dir, filename))

# # Delete unwanted wav files
# for name, filename in wav_files.items():
#     if name not in keep_names:
#         os.remove(os.path.join(wav_dir, filename))

# print(f"Kept {len(keep_names)} matching files, removed the rest.")

## 🚀 Run Full Training Pipeline (with Lip-Sync Loss)
This runs all phases:
1. Video processing (motion/eye/emotion features)
2. Bridge audio feature extraction
3. **Lip-sync feature preprocessing** (SyncNet embeddings, appearance features)
4. Data gathering
5. Training with lip-sync loss

Set `USE_LIP_SYNC=0` to disable lip-sync loss and train with motion-only losses.

In [ ]:
!pip uninstall -y torch torchaudio
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install --upgrade torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install --upgrade transformers accelerate huggingface_hub

In [ ]:
!python ./ditto-train/example/get_data_info_json_for_trainset_example.py

In [ ]:
%%bash
cd /workspace/moshi-ditto-training-lip-sync-mead-1000-demo-sir-try-phase-1-oom-error
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
HDTF_ROOT=/workspace/HDTF NUM_GPUS=4 USE_LIP_SYNC=1 \
LIP_SYNC_NUM_FRAMES=8 LIP_SYNC_USE_DELAY_AWARE=0 LIP_SYNC_MAX_SHIFT=0 \
bash ditto-train/scripts/prepare_and_train.sh


In [ ]:
%%bash
cd /workspace/moshi-ditto-training-lip-sync-mead-1000-demo-sir-try-phase-1-oom-error/ditto-train

echo "=== Step 1: Re-running preprocessing (8 GPUs, per-video output dirs) ==="
for gpu_id in 0 1; do
    CUDA_VISIBLE_DEVICES=$gpu_id python preprocess_lipsync_features.py \
        -i /workspace/HDTF/data_info.json \
        --syncnet_ckpt checkpoints/lipsync_expert.pth \
        --ditto_pytorch_path checkpoints/ditto_pytorch \
        --num_gpus 2 \
        --gpu_id $gpu_id \
        --num_frames 5 &
done
wait
echo "=== Preprocessing done ==="


In [ ]:
import json, os, glob, numpy as np, pickle
from tqdm import tqdm

DATA_INFO = "/workspace/HDTF/data_info.json"
DATA_LIST = "/workspace/HDTF/bridge_data_list_train.json"
DATA_PKL  = "/workspace/HDTF/bridge_preload.pkl"

with open(DATA_INFO) as f:
    data_info = json.load(f)

video_list = data_info.get('fps25_video_list', data_info.get('video_list', []))
N = len(video_list)
print(f"Total videos: {N}")

# ── STEP A: Build lipsync lists by scanning filesystem directly ──────────────
# Don't rely on shards — scan lipsync_features/ directory directly
LIPSYNC_ROOT = "/workspace/HDTF/lipsync_features"

lipsync_A_list   = [''] * N
lipsync_f_s_list = [''] * N
lipsync_xs_list  = [''] * N
lipsync_kpc_list = [''] * N
lipsync_sg_list  = [''] * N

found = 0
for i, vpath in enumerate(tqdm(video_list, desc="Scanning lipsync_features")):
    stem = os.path.splitext(os.path.basename(str(vpath)))[0]
    vdir = os.path.join(LIPSYNC_ROOT, stem)
    
    A_path   = os.path.join(vdir, 'lipsync_syncnet_A.npy')
    f_s_path = os.path.join(vdir, 'lipsync_f_s.npy')
    xs_path  = os.path.join(vdir, 'lipsync_x_s.npy')
    kpc_path = os.path.join(vdir, 'lipsync_kp_canonical.npy')
    sg_path  = os.path.join(vdir, 'lipsync_sim_gt.npy')
    
    if all(os.path.isfile(p) for p in [A_path, f_s_path, xs_path, kpc_path, sg_path]):
        lipsync_A_list[i]   = A_path
        lipsync_f_s_list[i] = f_s_path
        lipsync_xs_list[i]  = xs_path
        lipsync_kpc_list[i] = kpc_path
        lipsync_sg_list[i]  = sg_path
        found += 1

print(f"\nFound complete lipsync features: {found} / {N}")
print(f"Unique A paths: {len(set(x for x in lipsync_A_list if x))}")

# Spot-check
for a in lipsync_A_list:
    if a:
        arr = np.load(a)
        print(f"Sample A path : {a!r}")
        print(f"Sample A shape: {arr.shape}  (expect (N_win, 512))")
        break

# ── STEP B: Save to data_info.json ──────────────────────────────────────────
data_info['lipsync_A_list']            = lipsync_A_list
data_info['lipsync_f_s_list']          = lipsync_f_s_list
data_info['lipsync_x_s_list']          = lipsync_xs_list
data_info['lipsync_kp_canonical_list'] = lipsync_kpc_list
data_info['lipsync_sim_gt_list']       = lipsync_sg_list

with open(DATA_INFO, 'w') as f:
    json.dump(data_info, f)
print(f"\ndata_info.json updated with {found} lipsync entries")

# ── STEP C: Inject into PKL ──────────────────────────────────────────────────


In [ ]:
# ── STEP C: Inject correct paths into PKL ────────────────────────────────────
LP_npy_list = data_info.get('LP_npy_list', [])

def stem(path):
    b = os.path.basename(str(path))
    for s in ['_mtn.npy','_kps.npy','_aud.npy','.npy','.mp4','.wav']:
        if b.endswith(s): b = b[:-len(s)]
    return b.replace('_flip','')

lp_stem_to_idx = {stem(p): i for i, p in enumerate(LP_npy_list)}

with open(DATA_PKL, 'rb') as f:
    pkl = pickle.load(f)
with open(DATA_LIST) as f:
    entries = json.load(f)

arr_data_list = pkl[0]
patched = skipped = 0

for i, entry in enumerate(tqdm(entries, desc="Injecting into PKL")):
    j = lp_stem_to_idx.get(stem(entry.get('mtn', '')), -1)
    if j >= 0 and j < N and lipsync_A_list[j] and os.path.isfile(lipsync_A_list[j]):
        try:
            arr_data_list[i]['lipsync_f_s_path']          = lipsync_f_s_list[j]
            arr_data_list[i]['lipsync_x_s_path']          = lipsync_xs_list[j]
            arr_data_list[i]['lipsync_kp_canonical_path'] = lipsync_kpc_list[j]
            arr_data_list[i]['lipsync_A']                 = np.load(lipsync_A_list[j])
            arr_data_list[i]['lipsync_sim_gt']            = np.load(lipsync_sg_list[j])
            arr_data_list[i]['lipsync_valid']             = True
            patched += 1
        except Exception as ex:
            arr_data_list[i]['lipsync_valid'] = False
            skipped += 1
    else:
        arr_data_list[i]['lipsync_valid'] = False
        skipped += 1

pkl[0] = arr_data_list
with open(DATA_PKL, 'wb') as f:
    pickle.dump(pkl, f)

valid = sum(1 for s in arr_data_list if s.get('lipsync_valid', False))
print(f"Patched: {patched}  Skipped: {skipped}")
print(f"Final valid lipsync: {valid} / {len(arr_data_list)}")

# Final spot-check
for s in arr_data_list:
    if s.get('lipsync_valid'):
        fp = s['lipsync_f_s_path']
        arr = np.load(fp)
        print(f"\nSpot-check lipsync_f_s_path: {fp!r}")
        print(f"  File exists: {os.path.isfile(fp)}")
        print(f"  Shape: {arr.shape}  (expect (1, 32, 16, 64, 64))")
        print(f"  lipsync_A shape: {s['lipsync_A'].shape}  (expect (N_win, 512))")
        break


In [ ]:
# %%bash
# cd /workspace/moshi-ditto-training-lip-sync-mead-1000-demo-sir-try-phase-1-oom-error
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
# HDTF_ROOT=/workspace/HDTF NUM_GPUS=4 USE_LIP_SYNC=1 \
# LIP_SYNC_NUM_FRAMES=8 LIP_SYNC_USE_DELAY_AWARE=0 LIP_SYNC_MAX_SHIFT=0 \
# bash ditto-train/scripts/prepare_and_train.sh


In [ ]:
# # ── First: increase shared memory (run once in terminal) ──
# mount -o remount,size=16G /dev/shm 2>/dev/null || true
# df -h /dev/shm   # verify: should show 16G


In [32]:

# ── Then train ──
!NUM_GPUS=2 \
HDTF_ROOT=/workspace/HDTF \
BATCH_SIZE=512 \
NUM_WORKERS=4 \
MIXED_PRECISION=bf16 \
USE_LIP_SYNC=1 \
LIP_SYNC_NUM_FRAMES=5 \
LIP_SYNC_USE_DELAY_AWARE=0 \
LIP_SYNC_MAX_SHIFT=0 \
LIP_SYNC_EVERY_N=20 \
LIP_SYNC_WARMUP=5 \
LIP_SYNC_DEBUG=0 \
bash ditto-train/scripts/prepare_and_train.sh



═══════════════════════════════════════════════════════════════
   Bridge-Ditto Training Pipeline
═══════════════════════════════════════════════════════════════

   HDTF Root    : /workspace/HDTF
   GPUs         : 2
   Batch/GPU    : 512
   Epochs       : 1000
   Experiment   : bridge_ditto_hdtf_v1
   Audio dim    : 1103
   Motion dim   : 265
   Seq frames   : 80 (min_frames for gather: 81)
   Lip-sync     : ENABLED  λ1=0.05  λ2=0.02
   Lip frames   : 5  hard=clamp
   Delay-aware  : enabled=0  shift=±0  λ=0.05
   Lmk loss     : enabled=0  λ=0.02

═══════════════════════════════════════════════════════════════

[pipeline] PHASE 1: Video Processing (Ditto data preparation)...
[pipeline] Checking whether motion / emotion / eye / wav artifacts exist (sample paths from data_info.json)...
[check] OK: sample paths resolve on disk.
[pipeline] ✅  Feature files already present — skipping Phase 1.
[pipeline] ✅    (Delete outputs under /workspace/HDTF or re-run with missing .npy to regenerate.)


In [ ]:
# !HDTF_ROOT=/workspace/HDTF \
# NUM_GPUS=2 \
# USE_LIP_SYNC=1 \
# LIP_SYNC_NUM_FRAMES=16 \
# LIP_SYNC_HARD_MODE=clamp \
# LIP_SYNC_USE_DELAY_AWARE=1 \
# LIP_SYNC_MAX_SHIFT=3 \
# LIP_SYNC_DELAY_LAMBDA=0.05 \
# LIP_SYNC_LAMBDA1_WARMUP=10 \
# LIP_SYNC_DELAY_WARMUP=10 \
# bash ditto-train/scripts/prepare_and_train.sh

In [ ]:
!pip uninstall -y huggingface-hub
# !pip install "huggingface-hub<1.0.0,>=0.24"

In [ ]:
#_aKHzCnOPrapIEwfnuxyQUszisfBqaRrTXQ

In [ ]:
# !pip install -q huggingface_hub
from huggingface_hub import login

# Paste your token here
hf_token = ""
login(token=hf_token)



import os
from huggingface_hub import HfApi
from tqdm import tqdm

# --- CONFIGURATION ---

# Local folder you want to upload
local_folder = "./ditto-train/experiments/s2/bridge_ditto_hdtf_v1/ckpts"  # Replace with your folder path

# Hugging Face dataset repo info
repo_id = "Darknsu/MEAD_epoch_1000"  # Replace with your repo
repo_type = "dataset"

# Commit message
commit_message = "Upload entire folder with structure in one commit"

# --- UPLOAD FUNCTION ---
def upload_folder_single_commit(local_folder, repo_id, repo_type="dataset"):
    api = HfApi()

    print("📤 Uploading folder to Hugging Face with a single commit...")

    # Progress bar for local file count (optional)
    total_files = sum(len(files) for _, _, files in os.walk(local_folder))
    with tqdm(total=total_files, desc="Processing files") as pbar:
        for _, _, files in os.walk(local_folder):
            pbar.update(len(files))

    # Upload folder in a single commit
    api.upload_folder(
        folder_path=local_folder,
        repo_id=repo_id,
        repo_type=repo_type,
        commit_message=commit_message,
    )

    print("✅ Folder uploaded successfully in a single commit.")

# --- EXECUTE ---
upload_folder_single_commit(local_folder, repo_id, repo_type)

In [ ]:
# !HDTF_ROOT=/workspace/HDTF NUM_GPUS=8 USE_LIP_SYNC=1 bash ditto-train/scripts/prepare_and_train.sh